In [15]:
from typing import List
import pickle
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
from cv2.datasets import none
from narwhals import Boolean
from skimage.measure import fit
import time

In [23]:

def open_ghost_data(file_name: str):
	inputs = []
	expected_outputs = []
	with open(file_name, "r", encoding="utf-8") as file:
		for line in file:
			parsed_line = line.split()

			if len(parsed_line) >= 4:
				inputs.append([float(x) for x in parsed_line[0:3]])
				temp_output = [0, 0, 0, 0]
				temp_output[int(parsed_line[3]) - 1] = 1
				expected_outputs.append(temp_output)
	return np.array(inputs), np.array(expected_outputs)


def transform_mnist_data(y):
	output = []
	for data in y:
		temp = [0 for _ in range(10)]
		temp[int(data) - 1] = 1
		output.append(temp)
	return np.array(output)




In [24]:

train_inputs, train_outputs = open_ghost_data("../data/colors_train.txt")
test_inputs, test_outputs = open_ghost_data("../data/colors_test.txt")


In [25]:
import pandas
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
x_data, y_data = mnist.data, transform_mnist_data(mnist.target)
x_data = x_data / 255

In [26]:


x_train, x_test = x_data[:60000], x_data[60000:]
y_train, y_test = y_data[:60000], y_data[60000:]
print(f"x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"x_test: {x_test.shape}, y_test: {y_test.shape}")

x_train: (60000, 784), y_train: (60000, 10)
x_test: (10000, 784), y_test: (10000, 10)


In [27]:
class Module:
	def forward(self, x):
		raise NotImplementedError

	def backward(self, grad):
		raise NotImplementedError

	def adjust(self, eta):
		raise NotImplementedError


class LinearLayer(Module):
	def __init__(self, input_size: int, output_size: int, weights: None | np.ndarray = None,
				 biases: None | np.ndarray = None, min_weight: None | float = None, max_weight: None | float = None,
				 dropout_rate: float = 0.0):
		if weights is not None:
			if weights.shape != (output_size, input_size):
				raise ValueError("niezgodne wymiary macierzy wag")
			self.W = np.copy(weights)
		else:
			self.randomize_weights(min_weight, max_weight, output_size, input_size)

		if biases is not None:
			biases = np.atleast_2d(biases)
			if biases.shape != (1, output_size):
				raise ValueError("niezgodne wymiary macierzy biasow")
			self.b = np.copy(biases)
		else:
			self.b = None
		self.cache_X = None
		self.cache_grad = None
		self.is_training = False
		self.dropout_rate = dropout_rate
		self.dropout_mask = None

	def randomize_weights(self, min, max, output_size, input_size):
		self.W = np.random.uniform(min, max, (output_size, input_size))

	def forward(self, x):
		self.cache_X = np.atleast_2d(x)
		output = x @ self.W.T.copy()

		if self.b is not None:
			return output + self.b

		if self.dropout_rate > 0 and self.is_training:
			keep_rate = 1 - self.dropout_rate
			self.dropout_mask = np.random.binomial(1, keep_rate, output.shape) * (1 / keep_rate)
			output = output * self.dropout_mask
		else:
			self.dropout_mask = None
		return output

	def backward(self, grad):
		if self.dropout_mask is not None:
			grad = grad * self.dropout_mask
		self.cache_grad = grad
		return grad @ self.W

	def adjust(self, eta):
		grad_W = self.cache_grad.T @ self.cache_X
		self.W -= eta * grad_W
		if self.b is not None:
			grad_b = np.sum(self.cache_grad, axis=0, keepdims=True)
			self.b -= eta * grad_b


class ActivationLayer(Module):

	@staticmethod
	def relu(x: np.ndarray) -> np.ndarray:
		return np.maximum(x, 0)

	@staticmethod
	def relu_derivative(x: np.ndarray) -> np.ndarray:
		return np.where(x > 0, 1.0, 0)

	@staticmethod
	def sigmoid(x: np.ndarray) -> np.ndarray:
		return 1 / (1 + np.exp(-x))

	@staticmethod
	def sigmoid_derivative(x: np.ndarray) -> np.ndarray:
		return x * (1 - x)

	@staticmethod
	def tanh(x: np.ndarray) -> np.ndarray:
		return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

	@staticmethod
	def tanh_derivative(x: np.ndarray) -> np.ndarray:
		return 1 - x ** 2

	@staticmethod
	def softmax(x: np.ndarray) -> np.ndarray:
		exps = np.exp(x - np.max(x, axis=1, keepdims=True))
		return exps / np.sum(exps, axis=1, keepdims=True)

	ACTIVATION_FUNCTIONS = {
		'ReLU': (relu, relu_derivative),
		'Sigmoid': (sigmoid, sigmoid_derivative),
		'Tanh': (tanh, tanh_derivative),
		'Softmax': (softmax, None)
	}

	def __init__(self, activation_function):
		self.activation_name = activation_function
		self.activation_function = self.ACTIVATION_FUNCTIONS[activation_function][0]
		self.derivative_function = self.ACTIVATION_FUNCTIONS[activation_function][1]
		self.cache_output = None

	def forward(self, S):
		output = self.activation_function(S)
		self.cache_output = output
		return output

	def backward(self, grad):
		if self.activation_name == 'Softmax':
			return self.cache_output * (grad - np.sum(grad * self.cache_output, axis=1, keepdims=True))

		return grad * self.derivative_function(self.cache_output)

	def adjust(self, eta):
		pass


class NeuralNetworkException(Exception):
	def __init__(self, msg):
		self.messege = msg

	def get_message(self):
		return self.messege


class NeuralNetwork:

	def __init__(self, initial_input_size):
		self.layers: list[Module] = list()
		self.initial_input_size = initial_input_size
		self.last_layer_input_size = initial_input_size

	def add_layer(self, layer: Module):
		self.layers.append(layer)

	def save_model(self, filename: str):
		params = {}
		for i, layer in enumerate(self.layers):
			if isinstance(layer, LinearLayer):
				params[f'W_{i}'] = layer.W
				if layer.b is not None:
					params[f'b_{i}'] = layer.b
		np.savez(filename, **params)

	def load_model(self, filename: str):
		data = np.load(filename if filename.endswith('.npz') else filename + '.npz')
		for i, layer in enumerate(self.layers):
			if isinstance(layer, LinearLayer):
				layer.W = data[f'W_{i}']
				if f'b_{i}' in data:
					layer.b = data[f'b_{i}']

	def predict(self, x: np.ndarray | None):
		output = x
		for layer in self.layers:
			output = layer.forward(output)

		return output

	def train_step(self, x, y, learning_rate) -> float:

		prediction = self.predict(x)
		error = prediction - y
		loss = float(np.mean(np.square(error)))
		grad = (2.0 / error.size) * error
		for layer in reversed(self.layers):
			grad = layer.backward(grad)

		for layer in self.layers:
			layer.adjust(learning_rate)

		return loss

	def fit(self, x: np.ndarray, y, epochs, batch_size, learning_rate):
		self.switch_training(True)
		for epoch in range(epochs):
			indices = np.arange(x.shape[0])
			np.random.shuffle(indices)
			x_shuffled = x[indices]
			y_shuffled = y[indices]
			epoch_loss = 0.0
			batches_count = 0
			for batch in range(0, x.shape[0], batch_size):
				loss = self.train_step(x_shuffled[batch:batch + batch_size], y_shuffled[batch:batch + batch_size],
									   learning_rate)
				epoch_loss += loss
				batches_count += 1

		self.switch_training(False)

	pass

	def switch_training(self, state: bool):
		for layer in self.layers:
			if hasattr(layer, 'is_training'):
				layer.is_training = state

	def evaluate(self, x, y):
		data_size = y.shape[0]
		correct_predictions_count = 0
		false = []
		for i in range(data_size):
			predicted_num = np.argmax(self.predict(x[i:i + 1]))
			expected_num = np.argmax(y[i])
			if int(predicted_num) == int(expected_num):
				correct_predictions_count += 1
			else:
				false.append(i)
		return ((correct_predictions_count / data_size), false)




In [28]:
import time
import json
import numpy as np


class ExperimentRunner:
	def __init__(self, x_data, y_data, x_test, y_test, save_dir="../saves/"):
		self.x_data = x_data
		self.y_data = y_data
		self.x_test = x_test
		self.y_test = y_test
		self.save_dir = save_dir
		self.results = []

	def run_experiment(self, test_id: int, model: NeuralNetwork, test_params: dict):
		print(f"\n{'=' * 40}")
		print(f"TEST {test_id}")
		print(f"Data: {test_params['training_data_amount']}, Epochs: {test_params['iterations']}, "
			  f"Batch: {test_params['batch_size']}, LR: {test_params['alpha']}")
		print(f"{'=' * 40}")

		current_x_train = self.x_data[:test_params['training_data_amount']]
		current_y_train = self.y_data[:test_params['training_data_amount']]

		current_x_test = self.x_test[:test_params['test_data_amount']]
		current_y_test = self.y_test[:test_params['test_data_amount']]

		start_time = time.perf_counter()
		model.fit(current_x_train, current_y_train,
				  epochs=test_params['iterations'],
				  learning_rate=test_params['alpha'],
				  batch_size=test_params['batch_size'])
		end_time = time.perf_counter()
		training_time = end_time - start_time

		accuracy, _ = model.evaluate(current_x_test, current_y_test)

		print(f"Training Time: {training_time:.2f}s")
		print(f"Test Accuracy: {accuracy * 100:.2f}%")

		filename = f"{self.save_dir}model_test_{test_id}_data_{test_params['training_data_amount']}.npz"
		model.save_model(filename)
		print(f"Weights saved: {filename}")

		self._record_result(test_id, test_params, training_time, accuracy)

	def _record_result(self, test_id, test_params, time_taken, accuracy):
		result_entry = {
			"test_id": test_id,
			"accuracy": round(accuracy * 100, 2),
			"time_seconds": round(time_taken, 2),
			**test_params
		}
		self.results.append(result_entry)

	def save_summary_to_json(self, filename="experiment_results.json"):
		with open(filename, 'w') as f:
			json.dump(self.results, f, indent=4)
		print(f"\nZ: {filename}")

In [29]:
#zad 1 i 2

Wi_size = x_data.shape[1]
tests_parameters = [
	{
		'Wh_neurons': 40,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.005,
		'min_weight': -0.1,
		'max_weight': 0.1,
		'iterations': 100,
		'batch_size': 1
	}
	,
	{
		'Wh_neurons': 100,
		'training_data_amount': 5000,
		'test_data_amount': 10000,
		'alpha': 0.005,
		'min_weight': -0.1,
		'max_weight': 0.1,
		'iterations': 100,
		'batch_size': 1
	}
	,
	{
		'Wh_neurons': 100,
		'training_data_amount': 15000,
		'test_data_amount': 10000,
		'alpha': 0.005,
		'min_weight': -0.1,
		'max_weight': 0.1,
		'iterations': 100,
		'batch_size': 1
	}
	, #zadanie 2
	{
		'Wh_neurons': 40,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.1,
		'min_weight': -0.1,
		'max_weight': 0.1,
		'iterations': 100,
		'batch_size': 100
	}
	,
	{
		'Wh_neurons': 100,
		'training_data_amount': 5000,
		'test_data_amount': 10000,
		'alpha': 0.1,
		'min_weight': -0.1,
		'max_weight': 0.1,
		'iterations': 100,
		'batch_size': 100
	}
	,
	{
		'Wh_neurons': 100,
		'training_data_amount': 15000,
		'test_data_amount': 10000,
		'alpha': 0.1,
		'min_weight': -0.1,
		'max_weight': 0.1,
		'iterations': 100,
		'batch_size': 100
	}
]

runner = ExperimentRunner(x_data, y_data, x_test, y_test, save_dir="../saves/")

for i, test in enumerate(tests_parameters):

	if test['training_data_amount'] > 20000:
		continue

	test_network = NeuralNetwork(Wi_size)

	test_network.add_layer(LinearLayer(Wi_size, test['Wh_neurons'],
									   min_weight=test['min_weight'],
									   max_weight=test['max_weight'],
									   dropout_rate=0.5))
	test_network.add_layer(ActivationLayer("ReLU"))

	test_network.add_layer(LinearLayer(test['Wh_neurons'], y_test.shape[1],
									   min_weight=test['min_weight'],
									   max_weight=test['max_weight'],
									   dropout_rate=0.0))

	runner.run_experiment(test_id=i + 1, model=test_network, test_params=test)

runner.save_summary_to_json("wyniki_sieci_mnist_Z4_1-2.json")


TEST 1
Data: 1000, Epochs: 100, Batch: 1, LR: 0.005
Training Time: 30.94s
Test Accuracy: 85.64%
Weights saved: ../saves/model_test_1_data_1000.npz

TEST 2
Data: 5000, Epochs: 100, Batch: 1, LR: 0.005


KeyboardInterrupt: 

In [ ]:

Wi_size = x_data.shape[1]
tests_parameters = [
	{
		'Wh_neurons': 100,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.01,
		'min_weight': -0.01,
		'max_weight': 0.01,
		'iterations': 100,
		'batch_size': 100
	},

		{
		'Wh_neurons': 100,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.005,
		'min_weight': -0.01,
		'max_weight': 0.01,
		'iterations': 100,
		'batch_size': 100
	},
		{
		'Wh_neurons': 100,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.02,
		'min_weight': -0.01,
		'max_weight': 0.01,
		'iterations': 100,
		'batch_size': 100
	},
		{
		'Wh_neurons': 100,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.04,
		'min_weight': -0.01,
		'max_weight': 0.01,
		'iterations': 100,
		'batch_size': 100
	}
,
		{
		'Wh_neurons': 100,
		'training_data_amount': 1000,
		'test_data_amount': 10000,
		'alpha': 0.08,
		'min_weight': -0.01,
		'max_weight': 0.01,
		'iterations': 100,
		'batch_size': 100
	}

]

runner = ExperimentRunner(x_data, y_data, x_test, y_test, save_dir="../saves/")

for i, test in enumerate(tests_parameters):

	if test['training_data_amount'] > 20000:
		continue

	test_network = NeuralNetwork(Wi_size)

	test_network.add_layer(LinearLayer(Wi_size, test['Wh_neurons'],
									   min_weight=test['min_weight'],
									   max_weight=test['max_weight'],
									   dropout_rate=0.5))
	test_network.add_layer(ActivationLayer("Tanh"))

	test_network.add_layer(LinearLayer(test['Wh_neurons'], y_test.shape[1],
									   min_weight=test['min_weight'],
									   max_weight=test['max_weight'],
									   dropout_rate=0.0))
	test_network.add_layer(ActivationLayer("Softmax"))

	runner.run_experiment(test_id=i + 1, model=test_network, test_params=test)

runner.save_summary_to_json("wyniki_sieci_mnist_Z4_3.json")

